In [6]:
import random
import numpy as np
import streamlit as st
import math
from parameters import get_parameters
from global_func import odds_prob


# Initialize parameters
P = {}  # dict to restore probabilities
n = {}  # dict to restore counts
E = {}  # dict to restore effects
S = {}  # dict to restore supplies and capacities
OR = {}  # dict to restore odds ratio
M = {}  # dict to restore maternal outcomes

#known parameters
param = get_parameters()
LB_tot = np.array([23729, 18196, 20709, 5126])
P["highrisk_all"] = 0.26                                              # high risk pregnancies among all live births
P["ANC"] = 0.56
P["Home_noANC"] = 0.706

# Calibration criteria
P["home_all"] = param["home_all_target"]
P["l23_all"] = param["l23_all_target"]
P["l45_all"] = param["l45_all_target"]
P["home_anc"] = param["home_anc_target"] #0.07
P["l23_anc"] = 0.31
P["l45_anc"] = 0.62

In [7]:
random.seed(0)
def f_ANC_LB_effect(P_home_lowrisk, P_L23_highrisk, Sen_traditional, Spec_traditional):
    # Initialize parameters

    GA = param['GA_sequence']
    GA_n = len(GA)
    n["GA"] = param["GA_distribution"]
    P["GA"] = n["GA"] / np.sum(n["GA"])
    PT_mask = np.array([1] * 10 + [0] * 8, dtype=bool)
    FT_mask = ~PT_mask
    P["GA"][PT_mask] = P["GA"][PT_mask] * param["PT_scale"]  # scale P[GA|PT] to match kenya level
    P_mult = (1 - np.sum(P["GA"][PT_mask])) / np.sum(P["GA"][FT_mask])
    P["GA"][FT_mask] *= P_mult

    OR["ANC"] = param["OR_preterm_ANC"]
    Preterm_rate = np.sum(P["GA"][PT_mask])  # overall preterm rate

    preterm_anc_noanc = odds_prob(OR["ANC"], Preterm_rate, P["ANC"])
    P["GA_anc"] = P["GA"].copy()
    P_scale_anc = preterm_anc_noanc[0] / Preterm_rate
    P["GA_anc"][PT_mask] *= P_scale_anc
    P_mult = (1 - preterm_anc_noanc[0]) / np.sum(P["GA_anc"][FT_mask])
    P["GA_anc"][FT_mask] *= P_mult

    P["GA_noanc"] = P["GA"].copy()
    P_scale_noanc = preterm_anc_noanc[1] / Preterm_rate
    P["GA_noanc"][PT_mask] *= P_scale_noanc
    P_mult = (1 - preterm_anc_noanc[1]) / np.sum(P["GA_noanc"][FT_mask])
    P["GA_noanc"][FT_mask] *= P_mult

    E["Preterm_LMP"] = np.zeros(3)
    E["Preterm_LMP"][0], E["Preterm_LMP"][1] = param['E_Preterm_LMP'][0], param['E_Preterm_LMP'][1]
    E["Preterm_LMP"][2] = 1 - (E["Preterm_LMP"][0] + E["Preterm_LMP"][1])
    E["Postterm_LMP"] = np.zeros(3)
    E["Postterm_LMP"][0], E["Postterm_LMP"][1] = param['E_Postterm_LMP'][0], param['E_Postterm_LMP'][1]
    E["Postterm_LMP"][2] = 1 - (E["Postterm_LMP"][0] + E["Postterm_LMP"][1])

    P_home_noANC = P["Home_noANC"]
    P_l45_fac = 0.11 #assume is the probability of living cloest to L4/5
    P_fac_noANC = 1 - P_home_noANC
    P_L45_noANC = P_fac_noANC * P_l45_fac
    P_L23_noANC = P_fac_noANC - P_L45_noANC

    # Initialize counters
    n["highrisk"] = np.zeros(4)                                           #number of high-risk pregnancies by facility levels
    n["ANC"] = np.zeros(4)                                                #number of 4+ANC by facility levels
    n["LB_L"] = np.zeros(4)                                               #number of delivery location by facility levels
    M["PT"] = np.zeros(4)
    M["Postterm"] = np.zeros(4)
    M["GA"] = np.zeros((4, GA_n))  # GA distributions by facility levels

    for k_LB in range(np.sum(LB_tot)):
        i_risk = np.random.binomial(1, P["highrisk_all"])
        i_ANC = np.random.binomial(1, P["ANC"])
        if i_ANC:
            P_GA = P["GA_anc"]
        else:
            P_GA = P["GA_noanc"]
        i_jGA = np.searchsorted(np.cumsum(P_GA), np.random.rand())  # % actual index of GA
        i_GA = GA[i_jGA]  # % actual GA

        if i_GA < 37:
            i_term_status = 0   #% 0 = preterm, 1 = full term, 2 = postterm
        elif i_GA >= 43:
            i_term_status = 2
        else:
            i_term_status = 1

        if i_ANC == 0:
            i_loc = np.random.choice([0, 1, 2], p=[P_home_noANC, P_L23_noANC, P_L45_noANC])
        else:
            # Risk stratification
            if i_risk:
                i_risk_pred = 1 if np.random.random() < Sen_traditional else 0
            else:
                i_risk_pred = 0 if np.random.random() < Spec_traditional else 1

            # Gestational age estimation
            i_preterm_pred = 1 if np.random.random() < E["Preterm_LMP"][i_term_status] else 0
            i_postterm_pred = 1 if np.random.random() < E["Postterm_LMP"][i_term_status] else 0

            if i_risk_pred or i_preterm_pred == 1 or i_postterm_pred == 1:
                i_loc = np.random.choice([0, 1, 2], p=[0, P_L23_highrisk, 1 - P_L23_highrisk])
            else:
                i_loc = np.random.choice([0, 1, 2], p=[P_home_lowrisk, (1 - P_home_lowrisk) * 0.89, (1 - P_home_lowrisk) * 0.11])

        # assume among all live births at L45, 5126/(20709 + 5126) going to L5
        P_l5_l45 = 5126 / (20709 + 5126)
        if i_loc == 2 and np.random.random() < P_l5_l45:
            i_loc = 3

        # restore results
        n["highrisk"][i_loc] += i_risk
        n["ANC"][i_loc]      += i_ANC
        n["LB_L"][i_loc]     += 1
        M["GA"][i_loc, i_jGA] += 1
        M["PT"][i_loc] += i_term_status == 0
        M["Postterm"][i_loc] += i_term_status == 2

    # Predicted outcomes
    P_home_all = n["LB_L"][0] / np.sum(LB_tot)
    P_l23_all = n["LB_L"][1] / np.sum(LB_tot)
    P_l45_all = (n["LB_L"][2] + n["LB_L"][3]) / np.sum(LB_tot)
    P_home_anc = n["ANC"][0] / np.sum(n["ANC"])
    P_l23_anc = n["ANC"][1] / np.sum(n["ANC"])
    P_l45_anc = (n["ANC"][2] + n["ANC"][3]) / np.sum(n["ANC"])
    P_preterm_fac = np.sum(M["PT"][1:]) / np.sum(n["LB_L"][1:])
    P_preterm_home = M["PT"][0] / n["LB_L"][0]
    P_preterm_l23 = M["PT"][1] / n["LB_L"][1]
    P_preterm_l45 = (M["PT"][2] + M["PT"][3]) / n["LB_L"][2:4].sum()

    return P_home_all, P_l23_all, P_l45_all, P_home_anc, P_l23_anc, P_l45_anc, P_preterm_fac, P_preterm_home, P_preterm_l23, P_preterm_l45


In [8]:
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt

# Calibration targets
target_values = {
    "P_home_all": P["home_all"],
    "P_l23_all": P["l23_all"],
    "P_l45_all": P["l45_all"],
    "P_home_anc": P["home_anc"],
    "P_l23_anc": P["l23_anc"],
    "P_l45_anc": P["l45_anc"],
}

target_weights = {
    "P_home_all": (P["home_all"], 5),   # ±1% tolerance
    "P_l23_all":  (P["l23_all"], 5),
    "P_l45_all":  (P["l45_all"], 5),
    "P_home_anc": (P["home_anc"], 5),
    "P_l23_anc": (P["l23_anc"], 5),
    "P_l45_anc": (P["l45_anc"], 5),
}

def weighted_rmse(df_results, target_weights):
    weighted_errors = []
    for metric, (target, accepted_pct) in target_weights.items():
        simulated = df_results.get(metric, 0)
        error = simulated - target
        accepted_abs_error = target * (accepted_pct / 100)
        if accepted_abs_error == 0:  # safeguard
            continue
        normalized_error = error / accepted_abs_error
        weighted_errors.append(normalized_error**2)
    return math.sqrt(sum(weighted_errors) / len(weighted_errors))

# Define parameter space
param_space = [
    Real(0.1, 0.5, name='P_home_lowrisk'),
    Real(0.1, 0.4, name='P_L23_highrisk'),
    Real(0.6, 0.8, name='Sen_traditional'),
    Real(0.6, 0.8, name='Spec_traditional')
]

# Seeds for reproducibility
total_runs = 10
seeds = np.random.default_rng(2025).integers(low=0, high=1e6, size=total_runs)

# Track RMSE history
rmse_history = []

@use_named_args(param_space)
def objective(**params):
    # --- Constraint check: Sen must be greater than Spec ---
    if params['Sen_traditional'] <= params['Spec_traditional']:
        print("\n⚠️ Constraint violated: Sen_traditional <= Spec_traditional. Returning penalty.")
        objective.last_df_results = {}
        return 10.0  # Penalty loss

    results = []
    for run in range(total_runs):
        np.random.seed(seeds[run])
        res = f_ANC_LB_effect(
            params['P_home_lowrisk'],
            params['P_L23_highrisk'],
            params['Sen_traditional'],
            params['Spec_traditional']
        )
        results.append(res)

    # Average across runs
    mean_results = np.mean(results, axis=0)
    df_results = {
        "P_home_all": mean_results[0],
        "P_l23_all": mean_results[1],
        "P_l45_all": mean_results[2],
        "P_home_anc": mean_results[3],
        "P_l23_anc": mean_results[4],

    }

    # Save last results for early stopper
    objective.last_df_results = df_results

    # Weighted RMSE
    rmse = weighted_rmse(df_results, target_weights)

    # Print status
    print("\n--- Calibration Iteration ---")
    print("Parameters:")
    for k, v in params.items():
        print(f"  {k:22s}: {v:.4f}")
    print("\nTarget vs Simulated Outcomes:")
    for k in target_values:
        sim = df_results[k]
        target = target_values[k]
        error = sim - target
        rel_error = error / target
        print(f"  {k:22s} | Sim: {sim:.4f}, Target: {target:.4f}, RelErr: {rel_error:+.2%}")
    print(f"\nRMSE Loss: {rmse:.6f}")
    print("-----------------------------\n")

    return rmse

class EarlyStopper:
    def __init__(self, threshold_rmse=0.6, relative_threshold=0.05):
        self.threshold_rmse = threshold_rmse
        self.relative_threshold = relative_threshold
        self.best_loss = float('inf')
        self.best_params = None
        self.best_df_results = None

    def meets_relative_criteria(self, sim_results):
        for metric, (target, accepted_pct) in target_weights.items():
            sim = sim_results.get(metric, 0)
            rel_error = abs(sim - target) / target
            if rel_error > (accepted_pct / 100):
                return False
        return True

    def meets_param_constraints(self, param_list):
        if param_list is None:
            return False
        names = [d.name for d in param_space]
        p = dict(zip(names, param_list))
        return p['Sen_traditional'] > p['Spec_traditional']

    def __call__(self, *args, **kwargs):
        loss = objective(*args, **kwargs)
        rmse_history.append(loss)

        if loss < self.best_loss:
            self.best_loss = loss
            self.best_params = list(args[0]) if args else None
            self.best_df_results = getattr(objective, "last_df_results", {})

        if (self.best_loss < self.threshold_rmse
            and self.meets_relative_criteria(self.best_df_results)
            and self.meets_param_constraints(self.best_params)):
            print(f"\n⏹️ Early stopping: RMSE={self.best_loss:.6f} and Sen>Spec satisfied.")
            raise StopIteration

        return loss

def save_results(best_params, best_loss, rmse_history, param_space):
    param_names = [dim.name for dim in param_space]
    best_param_dict = dict(zip(param_names, best_params))
    best_param_dict["Best RMSE"] = best_loss
    df_best = pd.DataFrame([best_param_dict])
    df_best.to_csv("best_parameters_ANC_v6.csv", index=False)

    pd.DataFrame({"Iteration": range(1, len(rmse_history)+1), "RMSE": rmse_history}).to_csv("rmse_history_ANC_v6.csv", index=False)

    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(rmse_history)+1), rmse_history, marker='o')
    plt.xlabel("Iteration")
    plt.ylabel("RMSE Loss")
    plt.title("RMSE Loss Over Iterations")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("rmse_plot_ANC_v6.png")
    plt.close()

def optimize_model():
    early_stopper = EarlyStopper(threshold_rmse=0.6, relative_threshold=0.05)
    try:
        result = gp_minimize(
            func=early_stopper,
            dimensions=param_space,
            n_calls=1000,
            random_state=42,
            n_random_starts=20,
            acq_func="EI",
            verbose=True
        )
        print("\n✅ Optimization completed.")
        print("Best Parameters:", result.x)
        print("Best Loss:", result.fun)
        save_results(result.x, result.fun, rmse_history, param_space)
        return result

    except StopIteration:
        print("\n✅ Optimization stopped early.")
        print(f"Best RMSE achieved: {early_stopper.best_loss:.6f}")
        if early_stopper.best_params is not None:
            print("Best Parameters:")
            for name, val in zip([d.name for d in param_space], early_stopper.best_params):
                print(f"  {name:22s}: {val:.4f}")
            save_results(early_stopper.best_params, early_stopper.best_loss, rmse_history, param_space)
        return None

if __name__ == "__main__":
    optimize_model()


Iteration No: 1 started. Evaluating function at random point.

--- Calibration Iteration ---
Parameters:
  P_home_lowrisk        : 0.4186
  P_L23_highrisk        : 0.1550
  Sen_traditional       : 0.7559
  Spec_traditional      : 0.7194

Target vs Simulated Outcomes:
  P_home_all             | Sim: 0.4187, Target: 0.3500, RelErr: +19.64%
  P_l23_all              | Sim: 0.2941, Target: 0.3920, RelErr: -24.97%
  P_l45_all              | Sim: 0.2872, Target: 0.2580, RelErr: +11.30%
  P_home_anc             | Sim: 0.1923, Target: 0.0700, RelErr: +174.76%
  P_l23_anc              | Sim: 0.3193, Target: 0.3100, RelErr: +3.01%
  P_l45_anc              | Sim: 0.4883, Target: 0.6200, RelErr: -21.23%

RMSE Loss: 14.637104
-----------------------------

Iteration No: 1 ended. Evaluation done at random point.
Time taken: 7.3453
Function value obtained: 14.6371
Current minimum: 14.6371
Iteration No: 2 started. Evaluating function at random point.

--- Calibration Iteration ---
Parameters:
  P_home_

In [9]:
# --- Calibration Iteration ---
# Parameters:
#   P_l45_fac             : 0.1000
#   P_home_lowrisk        : 0.1671
#   P_L23_highrisk        : 0.3145
#   Sen_traditional       : 0.7542
#   Spec_traditional      : 0.6482
#
# Target vs Simulated Outcomes:
#   P_home_all             | Sim: 0.3504, Target: 0.3500, RelErr: +0.12%
#   P_l23_all              | Sim: 0.3926, Target: 0.3920, RelErr: +0.14%
#   P_l45_all              | Sim: 0.2570, Target: 0.2580, RelErr: -0.38%
#   P_home_anc             | Sim: 0.0705, Target: 0.0700, RelErr: +0.76%
#
# RMSE Loss: 0.435452
# -----------------------------
#
#
# ⏹️ Early stopping: RMSE=0.435452 and Sen>Spec satisfied.
#
# ✅ Optimization stopped early.
# Best RMSE achieved: 0.435452
# Best Parameters:
#   P_l45_fac             : 0.1000
#   P_home_lowrisk        : 0.1671
#   P_L23_highrisk        : 0.3145
#   Sen_traditional       : 0.7542
#   Spec_traditional      : 0.6482